# 01 · Train the phishing text classifier

**What this does.** Fits a TF-IDF + logistic regression pipeline on the Kaggle Phishing Email Dataset and saves it as `phish_clf.joblib`. The main notebook loads that file; it never retrains.

**Why.** The classifier is the proposal's baseline and becomes Guard's first evidence tool, `classify_email_text`. Its held-out scores are the component-level metric for the classifier alone.

**How it works.** Think of a mail clerk who never reads for meaning and only keeps a tally sheet. Words like *verify* and *suspended* add points, words like *agenda* subtract them. TF-IDF builds the tally of which words stand out; logistic regression learns the point values and turns the total into a probability.

All logic lives in `phishing_guard/classifier.py`. This notebook only calls it.

In [1]:
from phishing_guard.classifier import train_and_save, CSV_PATH, HOLDOUT_PATH
from phishing_guard.config import MODEL_PATH

print("corpus:", CSV_PATH.name)
print("model will be saved to:", MODEL_PATH.name)

corpus: phishing_email.csv
model will be saved to: phish_clf.joblib


## Train and score

Steps inside `train_and_save`: drop empty and duplicate rows, hold out 20% stratified by label with seed 42, fit on the other 80%, then report precision, recall, F1 on the phishing class, ROC-AUC, and the confusion matrix on the held-out 20%. It also saves three held-out phishing rows and three held-out legitimate rows to `data/kaggle_holdout.json` for reuse as test emails. Those rows were never seen in training.

In [2]:
metrics = train_and_save()
metrics

              precision    recall  f1-score   support

           0      0.986     0.982     0.984      7847
           1      0.984     0.987     0.985      8569

    accuracy                          0.985     16416
   macro avg      0.985     0.984     0.985     16416
weighted avg      0.985     0.985     0.985     16416

ROC-AUC: 0.999
[[7706  141]
 [ 112 8457]]


{'rows_after_cleaning': 82078,
 'train_rows': 65662,
 'test_rows': 16416,
 'precision_phishing': 0.984,
 'recall_phishing': 0.987,
 'f1_phishing': 0.985,
 'roc_auc': 0.999,
 'confusion_matrix': {'tn': 7706, 'fp': 141, 'fn': 112, 'tp': 8457}}

## Sanity check the saved model

Reload from disk the way the main notebook will, and score two toy emails with `top_words` so we can see which words drove each score.

In [3]:
from phishing_guard.classifier import load_classifier, top_words

clf = load_classifier()
samples = [
    "Your account has been suspended. Verify your password within 24 hours to restore access.",
    "Attached is the agenda for Thursday's Q4 vendor review meeting. See you in room 4B.",
]
for text in samples:
    prob = float(clf.predict_proba([text])[0][1])
    print(f"{prob:.2f}  {top_words(text)}  |  {text[:60]}")

0.95  ['account', 'verify', 'hours', 'suspended', '24 hours']  |  Your account has been suspended. Verify your password within
0.04  []  |  Attached is the agenda for Thursday's Q4 vendor review meeti


**What to expect.** The corpus mixes spam with phishing and much of it dates from the 2000s, so held-out scores will look strong. Scores on the 30 modern test emails in the main notebook will likely be weaker. Both get reported; the gap is the argument for putting an agent on top.